# Code Index For Independent Study

This page makes the tracked repository code inspectable as an independent-study record for Prof. Mgavi. It counts tracked scripts, configs, and docs by project area and links each area to the notebook page that exposes the code.

In [ ]:
from pathlib import Path
import ast
import html
import re
import subprocess
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SCRIPT_EXTS = (".py", ".ps1", ".sh")
CONFIG_EXTS = (".yaml", ".yml", ".json", ".toml", ".ini", ".cfg")
DOC_EXTS = (".md", ".txt", ".rst")


def run_git(args):
    result = subprocess.run(["git"] + list(args), cwd=ROOT, text=True, capture_output=True)
    if result.returncode != 0:
        return ""
    return result.stdout.strip()


def git_files():
    out = run_git(["ls-files"])
    return [line.strip().replace("\\", "/") for line in out.splitlines() if line.strip()]


def tracked_exists(path):
    return path.replace("\\", "/") in set(git_files())


def read_text(path, limit=None):
    rel = path.replace("\\", "/")
    if rel not in set(git_files()):
        return ""
    try:
        text = (ROOT / rel).read_text(encoding="utf-8", errors="replace")
    except Exception as exc:
        return f"Could not read {rel}: {exc}"
    return text if limit is None else text[:limit]


def line_count(path):
    text = read_text(path)
    return len(text.splitlines()) if text else 0


def last_commit(path):
    out = run_git(["log", "-1", "--format=%h|%cs|%s", "--", path])
    if not out:
        return {"commit": "not available", "date": "not available", "subject": "not available"}
    parts = out.split("|", 2)
    if len(parts) != 3:
        return {"commit": out, "date": "", "subject": ""}
    return {"commit": parts[0], "date": parts[1], "subject": parts[2]}


def header_docstring(path):
    text = read_text(path, limit=12000)
    if not text:
        return ""
    suffix = Path(path).suffix.lower()
    if suffix == ".py":
        try:
            doc = ast.get_docstring(ast.parse(text))
            if doc:
                return " ".join(doc.strip().split())[:280]
        except Exception:
            pass
    lines = text.splitlines()
    comments = []
    in_block = False
    for line in lines[:45]:
        stripped = line.strip()
        if suffix == ".ps1" and stripped.startswith("<#"):
            in_block = True
            continue
        if suffix == ".ps1" and stripped.endswith("#>"):
            in_block = False
            continue
        if in_block:
            comments.append(stripped.lstrip("#").strip())
        elif stripped.startswith("#"):
            comments.append(stripped.lstrip("#").strip())
        elif stripped and comments:
            break
        elif stripped and not comments:
            break
    return " ".join(c for c in comments if c)[:280]


def inferred_purpose(path):
    header = header_docstring(path)
    if header:
        return header
    stem = Path(path).stem
    tokens = re.sub(r"^\d+[a-zA-Z_]*_?", "", stem).replace("_", " ").replace("-", " ")
    return tokens[:1].upper() + tokens[1:] if tokens else "Purpose not available from tracked source."


def has_header(path):
    return "yes" if bool(header_docstring(path)) else "no obvious header/docstring"


def has_cli(path):
    text = read_text(path, limit=20000).lower()
    suffix = Path(path).suffix.lower()
    if suffix == ".py":
        return "yes" if ("argparse" in text or "click." in text or "if __name__ ==" in text or "typer." in text) else "no obvious CLI"
    if suffix == ".ps1":
        return "yes" if "param(" in text or "$args" in text else "no obvious CLI"
    return "not applicable"


def area_for(path):
    p = path.replace("\\", "/")
    if p.startswith("spatial_feature_identification_pipeline/"):
        return "Spatial Feature Identification"
    if p.startswith("prediction_modeling_pipeline/teacher_builder/"):
        return "Teacher Builder"
    if p.startswith("prediction_modeling_pipeline/model_training/"):
        return "Model Training"
    if p.startswith("prediction_modeling_pipeline/spatial_prediction_model_V2/"):
        return "Spatial Prediction Model V2"
    if p.startswith("prediction_modeling_pipeline/prediction_interpretation_model/"):
        return "Prediction Interpretation"
    if p.startswith("prediction_modeling_pipeline/spatial_transfer_inference_model/"):
        return "Spatial Transfer Inference"
    if p.startswith("scripts/") or p.startswith("data_manifest/") or p.endswith("project_profile.example.yaml"):
        return "Root Helpers, Configs, Manifests"
    return p.split("/", 1)[0]


def change_type(subject, files):
    low = subject.lower()
    if any(word in low for word in ["doc", "readme", "jupyter", "notebook", "pages"]):
        return "documentation/notebook"
    if any(word in low for word in ["workflow", "github", "pages", "ci", "requirements"]):
        return "infrastructure"
    if any(word in low for word in ["validate", "qc", "test", "smoke", "audit"]):
        return "validation/audit"
    if any(f.endswith(SCRIPT_EXTS) for f in files):
        return "code/scientific pipeline"
    if files and all(f.endswith(DOC_EXTS + CONFIG_EXTS) for f in files):
        return "documentation/config"
    return "mixed"


def md_table(rows, columns):
    if not rows:
        return "Not available in tracked repository."
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for row in rows:
        vals = []
        for col in columns:
            val = str(row.get(col, ""))
            val = val.replace("\n", " ").replace("|", "\\|")
            vals.append(val)
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def script_rows(prefixes, limit=None):
    files = [p for p in git_files() if p.endswith(SCRIPT_EXTS) and any(p.startswith(prefix) for prefix in prefixes)]
    rows = []
    selected = files if limit is None else files[:limit]
    for path in selected:
        commit = last_commit(path)
        rows.append({
            "path": f"`{path}`",
            "lines": line_count(path),
            "last commit": commit["commit"],
            "date": commit["date"],
            "purpose": inferred_purpose(path),
            "header/docstring": has_header(path),
            "CLI": has_cli(path),
        })
    return rows


def file_counts(prefixes):
    paths = [p for p in git_files() if any(p.startswith(prefix) for prefix in prefixes)]
    return {
        "scripts": sum(p.endswith(SCRIPT_EXTS) for p in paths),
        "configs": sum(p.endswith(CONFIG_EXTS) or "/configs/" in p for p in paths),
        "docs": sum(p.endswith(DOC_EXTS) or "readme" in Path(p).name.lower() for p in paths),
        "files": len(paths),
    }


def section_toc(path):
    text = read_text(path)
    headings = []
    for idx, line in enumerate(text.splitlines(), start=1):
        stripped = line.strip()
        if stripped.startswith("#") and len(stripped) < 140:
            headings.append((idx, stripped))
        elif re.match(r"^(def|class)\s+\w+", stripped):
            headings.append((idx, stripped))
    return headings[:30]


def code_details(path, mode="auto", max_full_lines=220, first=90, last=60):
    text = read_text(path)
    if not text:
        return f"**`{path}`** is not available in the tracked repository."
    lines = text.splitlines()
    lang = "python" if path.endswith(".py") else "powershell" if path.endswith(".ps1") else "text"
    escaped_path = html.escape(path)
    if mode == "full" or (mode == "auto" and len(lines) <= max_full_lines):
        body = html.escape(text)
        return f"<details><summary>Full source: <code>{escaped_path}</code> ({len(lines)} lines)</summary>\n\n<pre><code class='language-{lang}'>{body}</code></pre>\n</details>"
    toc = section_toc(path)
    toc_text = "\n".join(f"- line {line}: `{html.escape(title)}`" for line, title in toc) or "- No compact section outline detected."
    first_text = html.escape("\n".join(lines[:first]))
    last_text = html.escape("\n".join(lines[-last:]))
    return f"""<details><summary>Long script preview: <code>{escaped_path}</code> ({len(lines)} lines)</summary>

Tracked GitHub path: `{path}`

Section outline:

{toc_text}

First {min(first, len(lines))} lines:

<pre><code class='language-{lang}'>{first_text}</code></pre>

Last {min(last, len(lines))} lines:

<pre><code class='language-{lang}'>{last_text}</code></pre>
</details>"""


def display_markdown(text):
    display(Markdown(text))

In [ ]:
components = [
    {"pipeline/component": "Public Visium staging and root helpers", "prefixes": ["scripts/", "data_manifest/"], "page": "14_config_and_manifest_index"},
    {"pipeline/component": "Spatial Feature Identification Pipeline", "prefixes": ["spatial_feature_identification_pipeline/"], "page": "09_spatial_feature_identification_code"},
    {"pipeline/component": "Teacher Builder", "prefixes": ["prediction_modeling_pipeline/teacher_builder/"], "page": "10_teacher_builder_code"},
    {"pipeline/component": "Model Training", "prefixes": ["prediction_modeling_pipeline/model_training/"], "page": "11_model_training_code"},
    {"pipeline/component": "Spatial Prediction Model V2", "prefixes": ["prediction_modeling_pipeline/spatial_prediction_model_V2/"], "page": "12_spatial_prediction_v2_code"},
    {"pipeline/component": "Interpretation and Transfer", "prefixes": ["prediction_modeling_pipeline/prediction_interpretation_model/", "prediction_modeling_pipeline/spatial_transfer_inference_model/"], "page": "13_interpretation_and_transfer_code"},
]
rows = []
for component in components:
    counts = file_counts(component["prefixes"])
    scripts = [p for p in git_files() if p.endswith(SCRIPT_EXTS) and any(p.startswith(prefix) for prefix in component["prefixes"])]
    entry = [p for p in scripts if Path(p).name.startswith("00_") or Path(p).name.startswith("run_") or Path(p).name.startswith("download_")]
    rows.append({
        "pipeline/component": component["pipeline/component"],
        "script count": counts["scripts"],
        "config count": counts["configs"],
        "README/docs count": counts["docs"],
        "main entry-point scripts": ", ".join(f"`{p}`" for p in entry[:4]) or "not obvious from tracked files",
        "notebook pages that expose the code": f"`{component['page']}`",
    })
text = "## Repository Code Coverage Matrix\n\n" + md_table(rows, ["pipeline/component", "script count", "config count", "README/docs count", "main entry-point scripts", "notebook pages that expose the code"])
text += "\n\n## Reading Guide\n\nThe following pages expose tracked source code by pipeline area. Helper notebook cells are hidden or collapsed; project code inventories, code summaries, and selected source snippets are visible in the rendered book."
display_markdown(text)